In [0]:
# Arsh's accuracy experiment code

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from collections import Counter
from pyspark.sql.functions import udf

# =========================
# CONFIG
# =========================
OCR_TABLE = "tesseract_accuracy" # changed table name 
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# HELPERS
# =========================
def basename_expr(path_col):
    return F.lower(F.trim(F.regexp_extract(F.coalesce(path_col, F.lit("")), r"([^/]+)$", 1)))

def doc_id_from_digits_expr(path_col, min_len=6):
    base = basename_expr(path_col)
    return F.regexp_extract(base, rf"(\d{{{min_len},}})", 1)

def normalize_for_cer_expr(text_col):
    return F.regexp_replace(F.lower(F.coalesce(text_col, F.lit(""))), r"[^\p{L}\p{N}]+", "")

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER (
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================
def bow_f1_from_lists(a, b):
    if a is None or b is None or len(a) == 0 or len(b) == 0:
        return 0.0
    ca = Counter(a)
    cb = Counter(b)
    matched = 0
    for w in ca:
        if w in cb:
            matched += ca[w] if ca[w] < cb[w] else cb[w]
    prec = matched / float(len(a))
    rec  = matched / float(len(b))
    if (prec + rec) == 0.0:
        return 0.0
    return (2.0 * prec * rec) / (prec + rec)

def sliding_window_score(ocr_tokens, gt_tokens, window_size, step_size):
    if ocr_tokens is None or gt_tokens is None or len(ocr_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    ocr_windows = []
    i = 0
    while i < len(ocr_tokens):
        w = ocr_tokens[i:i+window_size]
        if len(w) > 0:
            ocr_windows.append(w)
        i += step_size

    if len(ocr_windows) == 0:
        return 0.0

    scores = []
    j = 0
    while j < len(gt_tokens):
        gt_w = gt_tokens[j:j+window_size]
        if len(gt_w) == 0:
            j += step_size
            continue

        best = 0.0
        for ow in ocr_windows:
            s = bow_f1_from_lists(ow, gt_w)
            if s > best:
                best = s

        scores.append(best)
        j += step_size

    if len(scores) == 0:
        return 0.0
    return float(sum(scores)) / float(len(scores))

@udf(returnType=DoubleType())
def sliding_window_udf(ocr_tokens, gt_tokens):
    return sliding_window_score(ocr_tokens, gt_tokens, TOKEN_WINDOW_SIZE, TOKEN_WINDOW_STEP)

df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


OCR rows: 11
GT rows : 11
Joined rows: 11


doc_id,ocr_path,gt_path
0000005183,dbfs:/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt
0000031553,dbfs:/FileStore/accuracy_experiment/rotated_0000031553.png,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt
0000014855,dbfs:/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt
0000022282,dbfs:/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt
0000008498,dbfs:/FileStore/accuracy_experiment/0000008498.DOCX,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt
0000022700,dbfs:/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt
0000005738,dbfs:/FileStore/accuracy_experiment/rotated_0000005738.png,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt
0000021535,dbfs:/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt
0000046076,dbfs:/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt
0000037576,dbfs:/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt


doc_id,ocr_path,gt_path,ocr_word_count,gt_word_count,matched_words,word_precision,word_recall,word_f1,edit_distance,gt_char_count,cer,char_accuracy,sliding_window_f1,run_ts
0000014855,dbfs:/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt,758.0,759.0,719.0,0.9485488126649076,0.9472990777338604,0.947923533289387,68.0,4055.0,0.016769420468557335,0.9832305795314427,0.9298886969327933,2026-02-03T01:05:46.649Z
0000022282,dbfs:/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt,230.0,230.0,219.0,0.9521739130434783,0.9521739130434783,0.9521739130434783,284.0,1440.0,0.19722222222222222,0.8027777777777778,0.8859999999999999,2026-02-03T01:05:46.649Z
0000034649,dbfs:/FileStore/accuracy_experiment/0000034649.png,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt,210.0,208.0,200.0,0.9523809523809523,0.9615384615384616,0.9569377990430622,356.0,1264.0,0.28164556962025317,0.7183544303797469,0.8066811909949165,2026-02-03T01:05:46.649Z
0000005183,dbfs:/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt,269.0,268.0,249.0,0.9256505576208178,0.9291044776119403,0.9273743016759777,659.0,1697.0,0.3883323512080141,0.6116676487919859,0.7523155130051683,2026-02-03T01:05:46.649Z
0000021535,dbfs:/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt,355.0,357.0,329.0,0.9267605633802817,0.9215686274509803,0.9241573033707865,929.0,2113.0,0.43965925224798863,0.5603407477520114,0.7325878136200716,2026-02-03T01:05:46.649Z
0000037576,dbfs:/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt,243.0,239.0,231.0,0.9506172839506173,0.9665271966527197,0.9585062240663901,732.0,1474.0,0.4966078697421981,0.5033921302578019,0.7290548780487806,2026-02-03T01:05:46.649Z
0000046076,dbfs:/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt,355.0,293.0,144.0,0.4056338028169014,0.49146757679180886,0.4444444444444445,1321.0,1716.0,0.7698135198135199,0.23018648018648014,0.39720430107526883,2026-02-03T01:05:46.649Z
0000022700,dbfs:/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt,148.0,428.0,137.0,0.9256756756756757,0.32009345794392524,0.4756944444444444,1899.0,2550.0,0.7447058823529412,0.2552941176470588,0.3348986774196858,2026-02-03T01:05:46.649Z
0000008498,dbfs:/FileStore/accuracy_experiment/0000008498.DOCX,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt,603.0,261.0,53.0,0.087893864013267,0.20306513409961685,0.12268518518518517,2549.0,1607.0,1.5861854387056626,0.0,0.15057775557775557,2026-02-03T01:05:46.649Z
0000005738,dbfs:/FileStore/accuracy_experiment/rotated_0000005738.png,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt,265.0,261.0,5.0,0.018867924528301886,0.019157088122605363,0.01901140684410646,1336.0,1607.0,0.8313627878033603,0.16863721219663974,0.028739887563416975,2026-02-03T01:05:46.649Z


doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
11,0.6121033532015443,0.9241573033707865,0.5237100939162216,0.7290548780487806,0.5987051155890991,0.4545844697477792,2026-02-03T01:05:48.229897Z


accuracy_results_per_doc
accuracy_results_summary


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
from pyspark.sql.functions import lit

overall_summary_schema = StructType([
    StructField("ocr_method", StringType(), nullable=False),
    StructField("doc_count", IntegerType(), nullable=False),
    StructField("avg_word_f1", FloatType()),
    StructField("median_word_f1", FloatType()),
    StructField("avg_sliding_window_f1", FloatType()),
    StructField("median_sliding_window_f1", FloatType()),
    StructField("avg_cer", FloatType()),
    StructField("avg_char_accuracy", FloatType()),
    StructField("run_ts", TimestampType(), nullable=False)
])

empty_df = spark.createDataFrame([], overall_summary_schema)

# ---------------------------
# Save as a managed table (overwrite if exists)
# ---------------------------
table_name = "overall_summary"
empty_df.write.mode("overwrite").saveAsTable(table_name)

In [0]:
summary = summary.withColumn("ocr_method", lit("tesseract")) # adding ocr method column

overall_summary = empty_df.unionByName(summary, allowMissingColumns=False)


display(summary)
display(overall_summary)

doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts,ocr_method
11,0.6121033532015443,0.9241573033707865,0.5237100939162216,0.7290548780487806,0.5987051155890991,0.4545844697477792,2026-02-03T01:05:58.130786Z,tesseract


ocr_method,doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
tesseract,11,0.6121033532015443,0.9241573033707865,0.5237100939162216,0.7290548780487806,0.5987051155890991,0.4545844697477792,2026-02-03T01:05:59.495144Z


In [0]:
# =========================
# CONFIG
# =========================
OCR_TABLE = "easyocr_exp" # changed table name again
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# HELPERS
# =========================
def basename_expr(path_col):
    return F.lower(F.trim(F.regexp_extract(F.coalesce(path_col, F.lit("")), r"([^/]+)$", 1)))

def doc_id_from_digits_expr(path_col, min_len=6):
    base = basename_expr(path_col)
    return F.regexp_extract(base, rf"(\d{{{min_len},}})", 1)

def normalize_for_cer_expr(text_col):
    return F.regexp_replace(F.lower(F.coalesce(text_col, F.lit(""))), r"[^\p{L}\p{N}]+", "")

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER 
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================
def bow_f1_from_lists(a, b):
    if a is None or b is None or len(a) == 0 or len(b) == 0:
        return 0.0
    ca = Counter(a)
    cb = Counter(b)
    matched = 0
    for w in ca:
        if w in cb:
            matched += ca[w] if ca[w] < cb[w] else cb[w]
    prec = matched / float(len(a))
    rec  = matched / float(len(b))
    if (prec + rec) == 0.0:
        return 0.0
    return (2.0 * prec * rec) / (prec + rec)

def sliding_window_score(ocr_tokens, gt_tokens, window_size, step_size):
    if ocr_tokens is None or gt_tokens is None or len(ocr_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    ocr_windows = []
    i = 0
    while i < len(ocr_tokens):
        w = ocr_tokens[i:i+window_size]
        if len(w) > 0:
            ocr_windows.append(w)
        i += step_size

    if len(ocr_windows) == 0:
        return 0.0

    scores = []
    j = 0
    while j < len(gt_tokens):
        gt_w = gt_tokens[j:j+window_size]
        if len(gt_w) == 0:
            j += step_size
            continue

        best = 0.0
        for ow in ocr_windows:
            s = bow_f1_from_lists(ow, gt_w)
            if s > best:
                best = s

        scores.append(best)
        j += step_size

    if len(scores) == 0:
        return 0.0
    return float(sum(scores)) / float(len(scores))

@udf(returnType=DoubleType())
def sliding_window_udf(ocr_tokens, gt_tokens):
    return sliding_window_score(ocr_tokens, gt_tokens, TOKEN_WINDOW_SIZE, TOKEN_WINDOW_STEP)

df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


OCR rows: 11
GT rows : 11
Joined rows: 11


doc_id,ocr_path,gt_path
0000022700,/dbfs/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt
0000014855,/dbfs/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt
0000021535,/dbfs/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt
0000034649,/dbfs/FileStore/accuracy_experiment/0000034649.png,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt
0000037576,/dbfs/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt
0000005183,/dbfs/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt
0000022282,/dbfs/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt
0000031553,/dbfs/FileStore/accuracy_experiment/rotated_0000031553.png,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt
0000046076,/dbfs/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt
0000008498,/dbfs/FileStore/accuracy_experiment/0000008498.png,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt


doc_id,ocr_path,gt_path,ocr_word_count,gt_word_count,matched_words,word_precision,word_recall,word_f1,edit_distance,gt_char_count,cer,char_accuracy,sliding_window_f1,run_ts
0000037576,/dbfs/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt,239.0,239.0,228.0,0.9539748953974896,0.9539748953974896,0.9539748953974896,457.0,1474.0,0.3100407055630936,0.6899592944369064,0.7983150183150183,2026-02-03T01:06:03.653683Z
0000034649,/dbfs/FileStore/accuracy_experiment/0000034649.png,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt,209.0,208.0,195.0,0.9330143540669856,0.9375,0.9352517985611511,443.0,1264.0,0.3504746835443038,0.6495253164556962,0.7723948883035802,2026-02-03T01:06:03.653683Z
0000021535,/dbfs/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt,361.0,357.0,310.0,0.8587257617728532,0.8683473389355743,0.8635097493036211,584.0,2113.0,0.27638428774254614,0.7236157122574538,0.7665620915032679,2026-02-03T01:06:03.653683Z
0000022282,/dbfs/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt,229.0,230.0,216.0,0.9432314410480349,0.9391304347826087,0.9411764705882353,334.0,1440.0,0.23194444444444445,0.7680555555555555,0.7348853439680957,2026-02-03T01:06:03.653683Z
0000005183,/dbfs/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt,267.0,268.0,239.0,0.8951310861423221,0.8917910447761194,0.8934579439252336,587.0,1697.0,0.34590453741897464,0.6540954625810254,0.7274407944996181,2026-02-03T01:06:03.653683Z
0000046076,/dbfs/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt,361.0,293.0,132.0,0.3656509695290859,0.45051194539249145,0.4036697247706422,1322.0,1716.0,0.7703962703962703,0.22960372960372966,0.31207487056949423,2026-02-03T01:06:03.653683Z
0000022700,/dbfs/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt,157.0,428.0,120.0,0.7643312101910829,0.2803738317757009,0.4102564102564102,1907.0,2550.0,0.7478431372549019,0.2521568627450981,0.3037037037037037,2026-02-03T01:06:03.653683Z
0000008498,/dbfs/FileStore/accuracy_experiment/0000008498.png,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt,166.0,261.0,16.0,0.0963855421686747,0.06130268199233716,0.07494145199063232,1228.0,1607.0,0.7641568139390168,0.2358431860609832,0.0702924012014921,2026-02-03T01:06:03.653683Z
0000014855,/dbfs/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt,390.0,759.0,6.0,0.015384615384615385,0.007905138339920948,0.010443864229765013,3906.0,4055.0,0.9632552404438964,0.03674475955610357,0.01913303286575176,2026-02-03T01:06:03.653683Z
0000005738,/dbfs/FileStore/accuracy_experiment/rotated_0000005738.png,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt,132.0,261.0,2.0,0.015151515151515152,0.007662835249042145,0.010178117048346057,1541.0,1607.0,0.9589296826384568,0.04107031736154321,0.007272727272727273,2026-02-03T01:06:03.653683Z


doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
11,0.4997145841883206,0.4102564102564102,0.4101886247457045,0.31207487056949423,0.607548261000142,0.39245173899985786,2026-02-03T01:06:05.272215Z


accuracy_results_per_doc
accuracy_results_summary


In [0]:
summary = summary.withColumn("ocr_method", lit("easyocr")) # adding ocr method column

overall_summary = overall_summary.unionByName(summary, allowMissingColumns=False)


display(summary)
display(overall_summary)

doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts,ocr_method
11,0.49971458418832065,0.4102564102564102,0.4101886247457044,0.31207487056949423,0.607548261000142,0.3924517389998579,2026-02-03T01:06:13.77445Z,easyocr


ocr_method,doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
tesseract,11,0.6121033532015443,0.9241573033707865,0.5237100939162215,0.7290548780487806,0.5987051155890992,0.45458446974777916,2026-02-03T01:06:15.23673Z
easyocr,11,0.4997145841883206,0.4102564102564102,0.4101886247457044,0.31207487056949423,0.607548261000142,0.3924517389998579,2026-02-03T01:06:15.23673Z


In [0]:
# edited this one to remove redunancies
# =========================
# CONFIG
# =========================
OCR_TABLE = "gpt5_exp"
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER (
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================


df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


OCR rows: 10
GT rows : 11
Joined rows: 10


doc_id,ocr_path,gt_path
0000022700,dbfs:/FileStore/tables/0000022700_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt
0000031553,dbfs:/FileStore/tables/0000031553_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt
0000008498,dbfs:/FileStore/tables/0000008498_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt
0000014855,dbfs:/FileStore/tables/0000014855_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt
0000021535,dbfs:/FileStore/tables/0000021535_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt
0000022282,dbfs:/FileStore/tables/0000022282_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt
0000034649,dbfs:/FileStore/tables/0000034649_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt
0000037576,dbfs:/FileStore/tables/0000037576_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt
0000005183,dbfs:/FileStore/tables/0000005183_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt
0000005738,dbfs:/FileStore/tables/0000005738_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt


doc_id,ocr_path,gt_path,ocr_word_count,gt_word_count,matched_words,word_precision,word_recall,word_f1,edit_distance,gt_char_count,cer,char_accuracy,sliding_window_f1,run_ts
0000037576,dbfs:/FileStore/tables/0000037576_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt,238.0,239.0,238.0,1.0,0.99581589958159,0.9979035639412999,24.0,1474.0,0.016282225237449117,0.9837177747625508,0.9949975949975951,2026-02-03T01:06:20.026803Z
0000031553,dbfs:/FileStore/tables/0000031553_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt,235.0,236.0,233.0,0.9914893617021276,0.9872881355932204,0.9893842887473461,4.0,1405.0,0.0028469750889679717,0.9971530249110321,0.9850127431254192,2026-02-03T01:06:20.026803Z
0000022282,dbfs:/FileStore/tables/0000022282_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt,229.0,230.0,228.0,0.9956331877729258,0.991304347826087,0.9934640522875816,1.0,1440.0,6.944444444444445E-4,0.9993055555555556,0.9831939736346517,2026-02-03T01:06:20.026803Z
0000005738,dbfs:/FileStore/tables/0000005738_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt,261.0,261.0,258.0,0.9885057471264368,0.9885057471264368,0.9885057471264368,6.0,1607.0,0.00373366521468575,0.9962663347853142,0.980119375573921,2026-02-03T01:06:20.026803Z
0000034649,dbfs:/FileStore/tables/0000034649_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt,207.0,208.0,207.0,1.0,0.9951923076923077,0.9975903614457832,106.0,1264.0,0.08386075949367089,0.9161392405063291,0.9686609686609687,2026-02-03T01:06:20.026803Z
0000005183,dbfs:/FileStore/tables/0000005183_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt,266.0,268.0,261.0,0.981203007518797,0.9738805970149254,0.9775280898876405,74.0,1697.0,0.043606364172068354,0.9563936358279317,0.9561242678889738,2026-02-03T01:06:20.026803Z
0000021535,dbfs:/FileStore/tables/0000021535_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt,354.0,357.0,354.0,1.0,0.9915966386554622,0.9957805907172996,142.0,2113.0,0.06720302886890676,0.9327969711310933,0.951872826626925,2026-02-03T01:06:20.026803Z
0000022700,dbfs:/FileStore/tables/0000022700_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt,281.0,428.0,280.0,0.99644128113879,0.6542056074766355,0.7898448519040903,877.0,2550.0,0.343921568627451,0.656078431372549,0.6458022430244652,2026-02-03T01:06:20.026803Z
0000014855,dbfs:/FileStore/tables/0000014855_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt,324.0,759.0,133.0,0.4104938271604938,0.17523056653491437,0.24561403508771928,3064.0,4055.0,0.7556103575832306,0.24438964241676941,0.22830318638994945,2026-02-03T01:06:20.026803Z
0000008498,dbfs:/FileStore/tables/0000008498_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt,430.0,261.0,50.0,0.11627906976744186,0.19157088122605365,0.14471780028943562,1713.0,1607.0,1.0659614187927815,0.0,0.15284922394678493,2026-02-03T01:06:20.026803Z


doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
10,0.8120333381434632,0.9885057471264368,0.7846936403869653,0.9561242678889738,0.23837208075236563,0.7682240611269124,2026-02-03T01:06:21.386799Z


accuracy_results_per_doc
accuracy_results_summary


In [0]:
summary = summary.withColumn("ocr_method", lit("gpt5")) # adding ocr method column

overall_summary = overall_summary.unionByName(summary, allowMissingColumns=False)


display(summary)
display(overall_summary)

doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts,ocr_method
10,0.8120333381434633,0.9885057471264368,0.7846936403869654,0.9561242678889738,0.23837208075236566,0.7682240611269124,2026-02-03T01:06:28.846789Z,gpt5


ocr_method,doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
tesseract,11,0.6121033532015443,0.9241573033707865,0.5237100939162215,0.7290548780487806,0.5987051155890992,0.45458446974777916,2026-02-03T01:06:30.067007Z
easyocr,11,0.4997145841883206,0.4102564102564102,0.4101886247457045,0.31207487056949423,0.607548261000142,0.3924517389998579,2026-02-03T01:06:30.067007Z
gpt5,10,0.8120333381434632,0.9885057471264368,0.7846936403869653,0.9561242678889738,0.23837208075236557,0.7682240611269124,2026-02-03T01:06:30.067007Z


In [0]:
# =========================
# CONFIG
# =========================
OCR_TABLE = "gpt4_exp"
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER (
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================


df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


OCR rows: 10
GT rows : 11
Joined rows: 10


doc_id,ocr_path,gt_path
0000022700,dbfs:/FileStore/tables/GPT4/0000022700_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt
0000031553,dbfs:/FileStore/tables/GPT4/0000031553_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt
0000008498,dbfs:/FileStore/tables/GPT4/0000008498_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt
0000014855,dbfs:/FileStore/tables/GPT4/0000014855_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt
0000021535,dbfs:/FileStore/tables/GPT4/0000021535_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt
0000022282,dbfs:/FileStore/tables/GPT4/0000022282_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt
0000034649,dbfs:/FileStore/tables/GPT4/0000034649_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt
0000037576,dbfs:/FileStore/tables/GPT4/0000037576_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt
0000005183,dbfs:/FileStore/tables/GPT4/0000005183_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt
0000005738,dbfs:/FileStore/tables/GPT4/0000005738_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt


doc_id,ocr_path,gt_path,ocr_word_count,gt_word_count,matched_words,word_precision,word_recall,word_f1,edit_distance,gt_char_count,cer,char_accuracy,sliding_window_f1,run_ts
0000037576,dbfs:/FileStore/tables/GPT4/0000037576_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt,238.0,239.0,238.0,1.0,0.99581589958159,0.9979035639412999,24.0,1474.0,0.016282225237449117,0.9837177747625508,0.9929975949975949,2026-02-03T01:06:36.300595Z
0000034649,dbfs:/FileStore/tables/GPT4/0000034649_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt,207.0,208.0,207.0,1.0,0.9951923076923077,0.9975903614457832,0.0,1264.0,0.0,1.0,0.9908831908831909,2026-02-03T01:06:36.300595Z
0000031553,dbfs:/FileStore/tables/GPT4/0000031553_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt,236.0,236.0,234.0,0.9915254237288136,0.9915254237288136,0.9915254237288136,10.0,1405.0,0.0071174377224199285,0.9928825622775801,0.9821313131313131,2026-02-03T01:06:36.300595Z
0000005183,dbfs:/FileStore/tables/GPT4/0000005183_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt,266.0,268.0,260.0,0.9774436090225563,0.9701492537313433,0.9737827715355805,33.0,1697.0,0.019446081319976428,0.9805539186800236,0.9488515406162464,2026-02-03T01:06:36.300595Z
0000022282,dbfs:/FileStore/tables/GPT4/0000022282_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt,228.0,230.0,224.0,0.9824561403508771,0.9739130434782609,0.9781659388646288,4.0,1440.0,0.002777777777777778,0.9972222222222222,0.9415517241379311,2026-02-03T01:06:36.300595Z
0000005738,dbfs:/FileStore/tables/GPT4/0000005738_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt,264.0,261.0,254.0,0.9621212121212122,0.9731800766283525,0.9676190476190476,48.0,1607.0,0.029869321717486,0.970130678282514,0.9024242424242424,2026-02-03T01:06:36.300595Z
0000021535,dbfs:/FileStore/tables/GPT4/0000021535_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt,349.0,357.0,347.0,0.994269340974212,0.9719887955182073,0.9830028328611897,30.0,2113.0,0.01419782300047326,0.9858021769995268,0.855528433334885,2026-02-03T01:06:36.300595Z
0000022700,dbfs:/FileStore/tables/GPT4/0000022700_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt,270.0,428.0,269.0,0.9962962962962963,0.6285046728971962,0.7707736389684814,911.0,2550.0,0.3572549019607843,0.6427450980392158,0.6222147248463038,2026-02-03T01:06:36.300595Z
0000014855,dbfs:/FileStore/tables/GPT4/0000014855_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt,221.0,759.0,80.0,0.36199095022624433,0.10540184453227931,0.16326530612244897,3149.0,4055.0,0.7765721331689273,0.22342786683107274,0.2823655913978495,2026-02-03T01:06:36.300595Z
0000008498,dbfs:/FileStore/tables/GPT4/0000008498_p1.txt,dbfs:/FileStore/tables/Ground_Truth/0000008498_DOCX___Ground_Truth.txt,430.0,261.0,50.0,0.11627906976744186,0.19157088122605365,0.14471780028943562,1714.0,1607.0,1.0665836963285626,0.0,0.15284922394678493,2026-02-03T01:06:36.300595Z


doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
10,0.7968346685376709,0.9737827715355805,0.7671797579716342,0.9024242424242424,0.22901013982338564,0.7776482298094706,2026-02-03T01:06:37.706844Z


accuracy_results_per_doc
accuracy_results_summary


In [0]:
summary = summary.withColumn("ocr_method", lit("gpt4")) # adding ocr method column

overall_summary = overall_summary.unionByName(summary, allowMissingColumns=False)


display(summary)
display(overall_summary)

doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts,ocr_method
10,0.7968346685376709,0.9737827715355805,0.7671797579716342,0.9024242424242424,0.22901013982338564,0.7776482298094706,2026-02-03T01:06:44.161831Z,gpt4


ocr_method,doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
tesseract,11,0.6121033532015443,0.9241573033707865,0.5237100939162215,0.7290548780487806,0.5987051155890992,0.45458446974777916,2026-02-03T01:06:45.319955Z
easyocr,11,0.4997145841883206,0.4102564102564102,0.4101886247457045,0.31207487056949423,0.6075482610001421,0.39245173899985786,2026-02-03T01:06:45.319955Z
gpt5,10,0.8120333381434632,0.9885057471264368,0.7846936403869653,0.9561242678889738,0.23837208075236557,0.7682240611269124,2026-02-03T01:06:45.319955Z
gpt4,10,0.7968346685376709,0.9737827715355805,0.7671797579716342,0.9024242424242424,0.22901013982338564,0.7776482298094706,2026-02-03T01:06:45.319955Z


In [0]:
# =========================
# CONFIG
# =========================
OCR_TABLE = "azure_ocr_exp"
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER (
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================


df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


OCR rows: 11
GT rows : 11
Joined rows: 11


doc_id,ocr_path,gt_path
0000037576,dbfs:/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt
0000034649,dbfs:/FileStore/accuracy_experiment/0000034649.png,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt
0000014855,dbfs:/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt
0000022282,dbfs:/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt
0000031553,dbfs:/FileStore/accuracy_experiment/rotated_0000031553.png,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt
0000005183,dbfs:/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt
0000021535,dbfs:/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt
0000046076,dbfs:/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt
0000022700,dbfs:/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt
0000005738,dbfs:/FileStore/accuracy_experiment/rotated_0000005738.png,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt


doc_id,ocr_path,gt_path,ocr_word_count,gt_word_count,matched_words,word_precision,word_recall,word_f1,edit_distance,gt_char_count,cer,char_accuracy,sliding_window_f1,run_ts
0000014855,dbfs:/FileStore/accuracy_experiment/0000014855.png,dbfs:/FileStore/tables/Ground_Truth/0000014855_PDF___Ground_Truth.txt,758.0,759.0,758.0,1.0,0.9986824769433466,0.999340804218853,0.0,4055.0,0.0,1.0,0.9976210031436744,2026-02-03T01:19:32.825119Z
0000022282,dbfs:/FileStore/accuracy_experiment/0000022282.png,dbfs:/FileStore/tables/Ground_Truth/0000022282_PDF___Ground_Truth.txt,229.0,230.0,229.0,1.0,0.9956521739130435,0.9978213507625272,36.0,1440.0,0.025,0.975,0.9871939736346516,2026-02-03T01:19:32.825119Z
0000005183,dbfs:/FileStore/accuracy_experiment/0000005183.png,dbfs:/FileStore/tables/Ground_Truth/0000005183_PDF___Ground_Truth.txt,268.0,268.0,266.0,0.9925373134328358,0.9925373134328358,0.9925373134328358,0.0,1697.0,0.0,1.0,0.9782898754991778,2026-02-03T01:19:32.825119Z
0000005738,dbfs:/FileStore/accuracy_experiment/rotated_0000005738.png,dbfs:/FileStore/tables/Ground_Truth/0000005738_PDF___Ground_Truth.txt,262.0,261.0,260.0,0.9923664122137404,0.9961685823754789,0.9942638623326959,4.0,1607.0,0.002489110143123833,0.9975108898568762,0.9662244842709405,2026-02-03T01:19:32.825119Z
0000034649,dbfs:/FileStore/accuracy_experiment/0000034649.png,dbfs:/FileStore/tables/Ground_Truth/0000034649_PDF___Ground_Truth.txt,208.0,208.0,207.0,0.9951923076923077,0.9951923076923077,0.9951923076923077,258.0,1264.0,0.20411392405063292,0.7958860759493671,0.82496632996633,2026-02-03T01:19:32.825119Z
0000037576,dbfs:/FileStore/accuracy_experiment/0000037576.png,dbfs:/FileStore/tables/Ground_Truth/0000037576_PDF___Ground_Truth.txt,239.0,239.0,238.0,0.99581589958159,0.99581589958159,0.99581589958159,435.0,1474.0,0.29511533242876525,0.7048846675712348,0.8082930402930403,2026-02-03T01:19:32.825119Z
0000021535,dbfs:/FileStore/accuracy_experiment/0000021535.png,dbfs:/FileStore/tables/Ground_Truth/0000021535_PDF___Ground_Truth.txt,356.0,357.0,352.0,0.9887640449438202,0.9859943977591037,0.9873772791023843,772.0,2113.0,0.36535731187884524,0.6346426881211548,0.7982238502238502,2026-02-03T01:19:32.825119Z
0000031553,dbfs:/FileStore/accuracy_experiment/rotated_0000031553.png,dbfs:/FileStore/tables/Ground_Truth/0000031553_PDF___Ground_Truth.txt,235.0,236.0,235.0,1.0,0.9957627118644068,0.9978768577494692,493.0,1405.0,0.3508896797153025,0.6491103202846975,0.7718296445338699,2026-02-03T01:19:32.825119Z
0000022700,dbfs:/FileStore/accuracy_experiment/0000022700.png,dbfs:/FileStore/tables/Ground_Truth/0000022700_PDF___Ground_Truth.txt,148.0,428.0,147.0,0.9932432432432432,0.34345794392523366,0.5104166666666667,1719.0,2550.0,0.6741176470588235,0.3258823529411765,0.42007022361690255,2026-02-03T01:19:32.825119Z
0000046076,dbfs:/FileStore/accuracy_experiment/rotated_0000046076.png,dbfs:/FileStore/tables/Ground_Truth/0000046076_PDF___Ground_Truth.txt,356.0,293.0,154.0,0.43258426966292135,0.5255972696245734,0.47457627118644063,1289.0,1716.0,0.7511655011655012,0.24883449883449882,0.4050961890132397,2026-02-03T01:19:32.825119Z


doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
11,0.8267224179154721,0.9942638623326959,0.734866450351955,0.8082930402930403,0.3140168212847586,0.6859831787152413,2026-02-03T01:19:35.176148Z


accuracy_results_per_doc
accuracy_results_summary


In [0]:
summary = summary.withColumn("ocr_method", lit("azure_ocr")) # adding ocr method column

overall_summary = overall_summary.unionByName(summary, allowMissingColumns=False)


display(summary)
display(overall_summary)

doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts,ocr_method
11,0.8267224179154721,0.9942638623326959,0.734866450351955,0.8082930402930403,0.3140168212847586,0.6859831787152413,2026-02-03T01:20:07.143823Z,azure_ocr


ocr_method,doc_count,avg_word_f1,median_word_f1,avg_sliding_window_f1,median_sliding_window_f1,avg_cer,avg_char_accuracy,run_ts
tesseract,11,0.6121033532015443,0.9241573033707865,0.5237100939162215,0.7290548780487806,0.5987051155890992,0.4545844697477792,2026-02-03T01:20:08.683819Z
easyocr,11,0.4997145841883206,0.4102564102564102,0.4101886247457045,0.31207487056949423,0.607548261000142,0.3924517389998579,2026-02-03T01:20:08.683819Z
gpt5,10,0.8120333381434632,0.9885057471264368,0.7846936403869653,0.9561242678889738,0.23837208075236563,0.7682240611269124,2026-02-03T01:20:08.683819Z
gpt4,10,0.7968346685376709,0.9737827715355805,0.7671797579716342,0.9024242424242424,0.22901013982338564,0.7776482298094706,2026-02-03T01:20:08.683819Z
azure_ocr,11,0.8267224179154721,0.9942638623326959,0.734866450351955,0.8082930402930403,0.3140168212847586,0.6859831787152414,2026-02-03T01:20:08.683819Z
